In [18]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import xgboost as xgb



df = pd.read_csv('lending_club_loan_two.csv')
print(df.shape)
df.head()

(396030, 27)


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,...,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies,address
0,10000.0,36 months,11.44,329.48,B,B4,Marketing,10+ years,RENT,117000.0,...,16.0,0.0,36369.0,41.8,25.0,w,INDIVIDUAL,0.0,0.0,"0174 Michelle Gateway\nMendozaberg, OK 22690"
1,8000.0,36 months,11.99,265.68,B,B5,Credit analyst,4 years,MORTGAGE,65000.0,...,17.0,0.0,20131.0,53.3,27.0,f,INDIVIDUAL,3.0,0.0,"1076 Carney Fort Apt. 347\nLoganmouth, SD 05113"
2,15600.0,36 months,10.49,506.97,B,B3,Statistician,< 1 year,RENT,43057.0,...,13.0,0.0,11987.0,92.2,26.0,f,INDIVIDUAL,0.0,0.0,"87025 Mark Dale Apt. 269\nNew Sabrina, WV 05113"
3,7200.0,36 months,6.49,220.65,A,A2,Client Advocate,6 years,RENT,54000.0,...,6.0,0.0,5472.0,21.5,13.0,f,INDIVIDUAL,0.0,0.0,"823 Reid Ford\nDelacruzside, MA 00813"
4,24375.0,60 months,17.27,609.33,C,C5,Destiny Management Inc.,9 years,MORTGAGE,55000.0,...,13.0,0.0,24584.0,69.8,43.0,f,INDIVIDUAL,1.0,0.0,"679 Luna Roads\nGreggshire, VA 11650"


In [19]:
df.info()
print("\nMissing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nLoan status distribution:\n", df['loan_status'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396030 entries, 0 to 396029
Data columns (total 27 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   loan_amnt             396030 non-null  float64
 1   term                  396030 non-null  object 
 2   int_rate              396030 non-null  float64
 3   installment           396030 non-null  float64
 4   grade                 396030 non-null  object 
 5   sub_grade             396030 non-null  object 
 6   emp_title             373103 non-null  object 
 7   emp_length            377729 non-null  object 
 8   home_ownership        396030 non-null  object 
 9   annual_inc            396030 non-null  float64
 10  verification_status   396030 non-null  object 
 11  issue_d               396030 non-null  object 
 12  loan_status           396030 non-null  object 
 13  purpose               396030 non-null  object 
 14  title                 394274 non-null  object 
 15  

In [20]:
# Keep only loans with a final outcome — drop 'Current', 'Late', etc.
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()

# Create binary target: 1 = defaulted (Charged Off), 0 = paid off fully
df['default'] = df['loan_status'].apply(lambda x: 1 if x == 'Charged Off' else 0)

print(df['default'].value_counts())
print(df['default'].value_counts(normalize=True))

default
0    318357
1     77673
Name: count, dtype: int64
default
0    0.803871
1    0.196129
Name: proportion, dtype: float64


In [21]:
# term: "36 months" -> 36
df['term'] = df['term'].str.extract('(\d+)').astype(int)

# int_rate / revol_util: strip % if stored as text
for col in ['int_rate', 'revol_util']:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.replace('%', '').astype(float)

# emp_length: convert to numeric years
def clean_emp_length(x):
    if pd.isna(x):
        return np.nan
    if '< 1' in x:
        return 0
    if '10+' in x:
        return 10
    return int(''.join(filter(str.isdigit, x)))

df['emp_length'] = df['emp_length'].apply(clean_emp_length)

# Drop high-cardinality / free-text columns not useful for modeling
df = df.drop(columns=['emp_title', 'title', 'address', 'issue_d', 'earliest_cr_line'], errors='ignore')

# Fill remaining numeric NaNs with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

print(df.isnull().sum()[df.isnull().sum() > 0])

<>:2: SyntaxWarning:

invalid escape sequence '\d'

<>:2: SyntaxWarning:

invalid escape sequence '\d'

/tmp/ipykernel_4374/1293072695.py:2: SyntaxWarning:

invalid escape sequence '\d'



Series([], dtype: int64)


In [22]:
import os
os.makedirs('charts', exist_ok=True)
grade_default = df.groupby('grade')['default'].mean().reset_index()
grade_default['default_rate_pct'] = grade_default['default'] * 100

fig = px.bar(grade_default, x='grade', y='default_rate_pct',
             title='Default Rate by Loan Grade',
             labels={'default_rate_pct': 'Default Rate (%)', 'grade': 'Grade'},
             color='default_rate_pct', color_continuous_scale='Reds')
fig.write_html('charts/default_rate_by_grade.html')
fig.show()

In [23]:
fig = px.histogram(df, x='int_rate', color='loan_status', barmode='overlay',
                    title='Interest Rate Distribution: Fully Paid vs Charged Off',
                    labels={'int_rate': 'Interest Rate (%)'}, opacity=0.6)
fig.write_html('charts/int_rate_distribution.html')
fig.show()

In [24]:
corr = df[numeric_cols].corr()
fig = px.imshow(corr, text_auto='.2f', aspect='auto',
                 title='Correlation Heatmap of Numeric Features',
                 color_continuous_scale='RdBu_r')
fig.write_html('charts/correlation_heatmap.html')
fig.show()

In [25]:
categorical_cols = ['grade', 'sub_grade', 'home_ownership', 'verification_status',
                     'purpose', 'application_type', 'initial_list_status']

df_model = df.copy()
le_dict = {}
for col in categorical_cols:
    if col in df_model.columns:
        le = LabelEncoder()
        df_model[col] = le.fit_transform(df_model[col].astype(str))
        le_dict[col] = le

# Drop the original text target, keep only 'default'
df_model = df_model.drop(columns=['loan_status'], errors='ignore')

print(df_model.dtypes)

loan_amnt               float64
term                      int64
int_rate                float64
installment             float64
grade                     int64
sub_grade                 int64
emp_length              float64
home_ownership            int64
annual_inc              float64
verification_status       int64
purpose                   int64
dti                     float64
open_acc                float64
pub_rec                 float64
revol_bal               float64
revol_util              float64
total_acc               float64
initial_list_status       int64
application_type          int64
mort_acc                float64
pub_rec_bankruptcies    float64
default                   int64
dtype: object


In [26]:
X = df_model.drop(columns=['default'])
y = df_model['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train default rate: {y_train.mean():.3f}, Test default rate: {y_test.mean():.3f}")

Train shape: (316824, 21), Test shape: (79206, 21)
Train default rate: 0.196, Test default rate: 0.196


In [27]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
y_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression Results")
print(classification_report(y_test, y_pred_lr))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_lr):.3f}")

Logistic Regression Results
              precision    recall  f1-score   support

           0       0.88      0.68      0.77     63671
           1       0.32      0.63      0.42     15535

    accuracy                           0.67     79206
   macro avg       0.60      0.65      0.60     79206
weighted avg       0.77      0.67      0.70     79206

ROC-AUC: 0.708


In [28]:
# scale_pos_weight handles class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost Results")
print(classification_report(y_test, y_pred_xgb))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_xgb):.3f}")

XGBoost Results
              precision    recall  f1-score   support

           0       0.89      0.67      0.76     63671
           1       0.33      0.67      0.44     15535

    accuracy                           0.67     79206
   macro avg       0.61      0.67      0.60     79206
weighted avg       0.78      0.67      0.70     79206

ROC-AUC: 0.724


In [29]:
cm = confusion_matrix(y_test, y_pred_xgb)
fig = px.imshow(cm, text_auto=True,
                 labels=dict(x="Predicted", y="Actual", color="Count"),
                 x=['Fully Paid', 'Charged Off'], y=['Fully Paid', 'Charged Off'],
                 title='XGBoost Confusion Matrix', color_continuous_scale='Blues')
fig.write_html('charts/confusion_matrix.html')
fig.show()

In [30]:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

fig = px.bar(importance, x='importance', y='feature', orientation='h',
             title='Top 15 Feature Importances — What Drives Loan Default',
             labels={'importance': 'Importance', 'feature': 'Feature'})
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.write_html('charts/feature_importance.html')
fig.show()

In [31]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_proba_xgb)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr_lr, y=tpr_lr, name=f'Logistic Regression (AUC={roc_auc_score(y_test, y_proba_lr):.3f})'))
fig.add_trace(go.Scatter(x=fpr_xgb, y=tpr_xgb, name=f'XGBoost (AUC={roc_auc_score(y_test, y_proba_xgb):.3f})'))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], line=dict(dash='dash'), name='Random'))
fig.update_layout(title='ROC Curve Comparison', xaxis_title='False Positive Rate', yaxis_title='True Positive Rate')
fig.write_html('charts/roc_curve.html')
fig.show()

In [35]:
import os
from google.colab import files

charts_dir = '/content/charts'
print(f"Downloading all files from {charts_dir}:")

for f_name in os.listdir(charts_dir):
    file_path = os.path.join(charts_dir, f_name)
    print(f"Downloading: {f_name}")
    files.download(file_path)


Downloading: feature_importance.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: confusion_matrix.html


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: correlation_heatmap.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>